# Simplified Knowledge Graph Visualization

This notebook generates a simplified, Property-Graph-style visualization of the music recommendation Knowledge Graph. It deliberately bypasses strict RDF reification to visually demonstrate edge weights (like listening counts) directly between the `Listener` and `Track` nodes, making it ideal for the project proposal presentation.

In [1]:
!pip install networkx pygraphviz
# Note: pygraphviz requires Graphviz to be installed on your system (e.g., `sudo apt-get install graphviz graphviz-dev` on Linux or `brew install graphviz` on Mac)

In [3]:
import io
import pygraphviz as pgv
from IPython.display import Image, display

SAVE_OUTPUT   = True
output_filename = "mrc_kg_simple_gnn.png"

color_map = {
    "Listener":  "#FF9999",
    "Track":     "#99CCFF",
    "Artist":    "#99FF99",
    "Genre":     "#FFCC99",
    "Metadata":  "#B0D4FF",
    "Feature":   "#E0E0E0",
    "External":  "#FFD700",
    "Hierarchy": "#D8BFD8",
}

A = pgv.AGraph(directed=True, strict=False)
A.graph_attr.update(
    rankdir="LR",
    dpi="300",
    nodesep="0.8",
    ranksep="2.0",
    fontname="Helvetica",
    compound="true",
    splines="polyline",
)

def add_node(graph, node_id, label, ntype, shape="box"):
    graph.add_node(
        node_id,
        label=label,
        style="filled, rounded",
        shape=shape,
        fillcolor=color_map.get(ntype, "white"),
        fontname="Helvetica-Bold",
        fontsize="13",
        margin="0.28,0.14",
    )

def make_html_label(label):
    label = label.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
    lines = label.split("\n")
    rows = "".join(f'<TR><TD ALIGN="CENTER">{l}</TD></TR>' for l in lines)
    return (
        f'<<TABLE BORDER="0" CELLBORDER="0" CELLSPACING="1" CELLPADDING="4" '
        f'BGCOLOR="white">{rows}</TABLE>>'
    )

def add_edge(g, src, dst, label, bold=False, color="#444444", dashed=False):
    g.add_edge(
        src, dst,
        label=make_html_label(label),
        fontname="Helvetica",
        fontsize="11",
        color=color,
        fontcolor="#111111",
        penwidth="2.8" if bold else "1.5",
        style="dashed" if dashed else "solid",
    )

# ── Core nodes ────────────────────────────────────────────────────────────────
add_node(A, "listener", "mrc:Listener\n(e.g. user_001)\n───────────────\nNode feat: none\n(learned embedding)", "Listener")
add_node(A, "track",    "mrc:MSDTrack\n(e.g. TRAAAGR128F)\n───────────────\nNode feat: ae_0…ae_127\n(128-dim AE embedding)", "Track")
add_node(A, "artist",   "mo:Performer\n(e.g. Radiohead)\n───────────────\nNode feat: none\n(learned embedding)", "Artist")

# ── Cluster: Genre Hierarchy ──────────────────────────────────────────────────
# Shows skos:broader chain: Electronic > Trip-Hop > Portishead-style
genre_cluster = A.add_subgraph(
    name="cluster_genre",
    label="Genre Hierarchy  (skos:broader chain, Wikidata-resolved)",
    style="dashed", color="#AA6633", fontcolor="#AA6633",
    fontname="Helvetica-Bold", fontsize="12", bgcolor="#FFF5E6",
)
add_node(genre_cluster, "electronic",  "Electronic\n(Q9778)\n[root concept]",          "Genre")
add_node(genre_cluster, "trip_hop",    "Trip-Hop\n(Q207648)\n[mid-level]",             "Genre")
add_node(genre_cluster, "indie_rock",  "Indie Rock\n(Q183further)\n[leaf concept]",    "Genre")
add_node(genre_cluster, "rock",        "Rock\n(Q11399)\n[root concept]",               "Genre")

genre_cluster.add_edge("trip_hop",   "electronic", label=make_html_label("skos:broader"),
    fontname="Helvetica", fontsize="11", fontcolor="#AA6633",
    color="#AA6633", penwidth="1.5", style="dashed")
genre_cluster.add_edge("indie_rock", "rock",       label=make_html_label("skos:broader"),
    fontname="Helvetica", fontsize="11", fontcolor="#AA6633",
    color="#AA6633", penwidth="1.5", style="dashed")

# ── Cluster: Instrument Hierarchy ─────────────────────────────────────────────
# MIDI program ID → Wikidata subclass chain
inst_cluster = A.add_subgraph(
    name="cluster_instruments",
    label="Instrument Hierarchy  (MIDI program ID → Wikidata rdfs:subClassOf)",
    style="dashed", color="#8B008B", fontcolor="#8B008B",
    fontname="Helvetica-Bold", fontsize="12", bgcolor="#FAF0FF",
)
add_node(inst_cluster, "chordophone",  "Chordophone\n(Q1055744)\n[root class]",    "External")
add_node(inst_cluster, "str_inst",     "String Instrument\n(Q1798603)\n[mid-level]","External")
add_node(inst_cluster, "guitar",       "Guitar\n(Q6607)\n[leaf, MIDI prog 24-31]", "External")
add_node(inst_cluster, "violin",       "Violin\n(Q8355)\n[leaf, MIDI prog 40]",    "External")

inst_cluster.add_edge("str_inst", "chordophone", label=make_html_label("rdfs:subClassOf"),
    fontname="Helvetica", fontsize="11", fontcolor="#8B008B",
    color="#8B008B", penwidth="1.5", style="dashed")
for child in ["guitar", "violin"]:
    inst_cluster.add_edge(child, "str_inst", label=make_html_label("rdfs:subClassOf"),
        fontname="Helvetica", fontsize="11", fontcolor="#8B008B",
        color="#8B008B", penwidth="1.5", style="dashed")

# ── Cluster: MSD Metadata (attached directly to Track in simple KG) ──────────
meta_cluster = A.add_subgraph(
    name="cluster_metadata",
    label="MSD Derived Metadata  (attached directly to Track in simple variant)",
    style="dashed", color="#4A90E2", fontcolor="#4A90E2",
    fontname="Helvetica-Bold", fontsize="12", bgcolor="#F0F8FF",
)
add_node(meta_cluster, "decade",  "mrc:Decade\n(e.g. 1990s)\n[Wikidata Q34653]",     "Metadata")
add_node(meta_cluster, "key",     "mrc:Key\n(e.g. Key/F)\n[12 named individuals]",   "Metadata")
add_node(meta_cluster, "mode",    "mrc:Mode\n(e.g. MajorMode)\n[2 named individuals]","Metadata")

# ── Cluster: jSymbolic-derived Feature Nodes ─────────────────────────────────
feat_cluster = A.add_subgraph(
    name="cluster_features",
    label="jSymbolic-derived Feature Concepts  (categorical, named individuals)",
    style="dashed", color="#888888", fontcolor="#888888",
    fontname="Helvetica-Bold", fontsize="12", bgcolor="#F9F9F9",
)
add_node(feat_cluster, "tempo_class", "mrc:TempoClass\n(e.g. Allegro)\n[120–156 BPM]", "Feature")

# ── Backbone edges ────────────────────────────────────────────────────────────
# Listening edge — simple variant: direct triple + listenCount as edge weight
add_edge(A, "listener", "track",
    "mrc:listenedTo\n─────────────────\nedge weight: listenCount\n(e.g. 47 plays)\nfiltered: count ≥ 5",
    bold=True, color="#D9534F")

# Track ↔ Artist
add_edge(A, "track",  "artist",    "mo:performer")
add_edge(A, "artist", "track",     "mrc:performed\n(inverse)")

# Artist → Genre  (simple: direct edge, no blank node)
# Weight shown as edge annotation — in HeteroData this becomes an edge_attr
add_edge(A, "artist", "trip_hop",
    "mrc:hasGenre\n─────────────\nedge weight: term_weight\n(e.g. 0.85, from MSD artist_terms)",
    color="#AA6633")
add_edge(A, "artist", "indie_rock",
    "mrc:hasGenre\n─────────────\nedge weight: term_weight\n(e.g. 0.42)",
    color="#AA6633")

# Track → Metadata (simple: key/mode directly on Track, not Performance)
add_edge(A, "track", "key",         "mrc:hasKey\n(simple variant:\ndirect on Track)")
add_edge(A, "track", "mode",        "mrc:hasMode\n(simple variant:\ndirect on Track)")
add_edge(A, "track", "decade",      "mrc:releaseDecade")
add_edge(A, "track", "tempo_class", "mrc:hasTempoClass")

# Track → Instruments
add_edge(A, "track", "guitar", "mo:instrument\n(MIDI prog 25)", color="#8B008B")
add_edge(A, "track", "violin", "mo:instrument\n(MIDI prog 40)", color="#8B008B")

# ── GNN node type legend ──────────────────────────────────────────────────────
gnn_label = (
    '<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" CELLPADDING="5" BGCOLOR="white">'
    '<TR><TD COLSPAN="3" BGCOLOR="#CCCCCC"><B>HeteroData Node Types seen by HGT</B></TD></TR>'
    '<TR><TD BGCOLOR="#DDDDDD"><B>Node type</B></TD><TD BGCOLOR="#DDDDDD"><B>Count (approx)</B></TD>'
    '<TD BGCOLOR="#DDDDDD"><B>Initial features</B></TD></TR>'
    '<TR><TD>user</TD><TD>~3 k</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>track</TD><TD>~7 k</TD><TD>128-d AE embedding</TD></TR>'
    '<TR><TD>artist</TD><TD>~4 k</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>genre</TD><TD>~300</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>instrument</TD><TD>~128</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>key / mode</TD><TD>12 / 2</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>tempo_class</TD><TD>10</TD><TD>learned (nn.Embedding)</TD></TR>'
    '<TR><TD>decade</TD><TD>~8</TD><TD>learned (nn.Embedding)</TD></TR>'
    '</TABLE>>'
)
A.add_node("gnn_legend", label=gnn_label, shape="none", margin="0",
           fontname="Helvetica", fontsize="11")

# ── Ontology prefix legend ────────────────────────────────────────────────────
prefix_label = (
    '<<TABLE BORDER="0" CELLBORDER="1" CELLSPACING="0" CELLPADDING="4" BGCOLOR="white">'
    '<TR><TD COLSPAN="2" BGCOLOR="#E0E0E0"><B>Ontology Prefixes</B></TD></TR>'
    '<TR><TD>mrc:</TD><TD>Music Recommendation Ontology (ours)</TD></TR>'
    '<TR><TD>mo:</TD><TD>Music Ontology</TD></TR>'
    '<TR><TD>skos:</TD><TD>Simple Knowledge Organization System</TD></TR>'
    '<TR><TD>rdfs:</TD><TD>RDF Schema</TD></TR>'
    '<TR><TD>wd:</TD><TD>Wikidata entity namespace</TD></TR>'
    '</TABLE>>'
)
A.add_node("prefix_legend", label=prefix_label, shape="none", margin="0",
           fontname="Helvetica", fontsize="11")

# ── Render ────────────────────────────────────────────────────────────────────
if SAVE_OUTPUT:
    A.draw(output_filename, prog="dot")
    display(Image(filename=output_filename))
    print(f"Graph saved as '{output_filename}'")
else:
    buf = io.BytesIO()
    buf.write(A.draw(format="png", prog="dot"))
    buf.seek(0)
    display(Image(data=buf.read()))

Graph saved as 'mrc_kg_simple_gnn.png'
